# 3. Public Access — Azure Firewall, App Gateway + WAF, Front Door, DDoS

## Azure Firewall

Centralized, managed, stateful firewall. Deployed in a hub VNet with spokes routed through it.

### Firewall SKUs

| | Standard | Premium |
|-|----------|----------|
| L3-L7 filtering | ✅ | ✅ |
| FQDN filtering | ✅ | ✅ |
| Threat intelligence | ✅ | ✅ |
| **TLS inspection** | ❌ | ✅ |
| **IDPS** (intrusion detection) | ❌ | ✅ |
| **URL filtering** (full URL, not just FQDN) | ❌ | ✅ |
| **Web categories** | ❌ | ✅ |
| Cost | ~$912/month | ~$1,825/month |

### Firewall rule types (processed in order)

| Rule type | What it filters | Example |
|-----------|----------------|----------|
| **NAT rules** | Inbound DNAT (port forwarding) | Forward public port 443 → internal VM |
| **Network rules** | L3/L4 (IP, port, protocol) | Allow 10.0.0.0/8 → SQL port 1433 |
| **Application rules** | L7 FQDN/URL | Allow *.microsoft.com, block *.gambling.com |

In [ ]:
# Simulate Azure Firewall rule evaluation
FW_RULES = {
    'nat': [
        {'name': 'WebInbound', 'src': '*', 'dst_port': 443, 'translated_ip': '10.0.1.10', 'translated_port': 443},
    ],
    'network': [
        {'name': 'AllowAppToSQL', 'src': '10.0.2.0/24', 'dst': '10.0.3.0/24', 'port': 1433, 'action': 'Allow'},
        {'name': 'AllowDNS',     'src': '10.0.0.0/8',   'dst': '*',           'port': 53,   'action': 'Allow'},
    ],
    'application': [
        {'name': 'AllowMicrosoft',  'src': '10.0.0.0/8', 'fqdn': '*.microsoft.com',  'protocol': 'HTTPS', 'action': 'Allow'},
        {'name': 'AllowGitHub',     'src': '10.0.0.0/8', 'fqdn': '*.github.com',     'protocol': 'HTTPS', 'action': 'Allow'},
        {'name': 'BlockGambling',   'src': '10.0.0.0/8', 'fqdn': '*.gambling.com',   'protocol': 'HTTPS', 'action': 'Deny'},
        {'name': 'AllowUbuntuAPT',  'src': '10.0.0.0/8', 'fqdn': 'archive.ubuntu.com', 'protocol': 'HTTP', 'action': 'Allow'},
    ],
}

print('=== Azure Firewall Rule Sets ===')
for rule_type, rules in FW_RULES.items():
    print(f'\n--- {rule_type.upper()} rules (processed {"first" if rule_type == "nat" else "second" if rule_type == "network" else "last"}) ---')
    for r in rules:
        if rule_type == 'nat':
            print(f'  {r["name"]}: *:{r["dst_port"]} → {r["translated_ip"]}:{r["translated_port"]}')
        elif rule_type == 'network':
            print(f'  {r["name"]}: {r["src"]} → {r["dst"]}:{r["port"]} [{r["action"]}]')
        else:
            print(f'  {r["name"]}: {r["src"]} → {r["fqdn"]} [{r["action"]}]')

print('\n💡 Processing order: NAT → Network → Application → Implicit deny')

### Azure Firewall Manager

Manages firewall policies across multiple firewalls and regions:
- **Firewall policies** are hierarchical (parent → child inheritance)
- **Secured virtual hubs** integrate firewall with Virtual WAN
- Central management for global enterprises

```bash
# Create a firewall policy
az network firewall policy create -g rg-hub -n fw-policy-global \
  --sku Premium --threat-intel-mode Deny

# Create child policy inheriting from parent
az network firewall policy create -g rg-hub -n fw-policy-westeurope \
  --base-policy fw-policy-global --sku Premium
```

---
## Application Gateway + WAF

| Feature | Application Gateway | Azure Firewall |
|---------|-------------------|----------------|
| Layer | L7 (HTTP/HTTPS only) | L3-L7 (any protocol) |
| SSL termination | ✅ | ✅ (Premium only) |
| WAF | ✅ (built-in) | ❌ (different service) |
| Load balancing | ✅ (L7) | ❌ |
| URL path routing | ✅ | ❌ |
| Use case | Web app frontend | Network security gateway |

### WAF policies

WAF can run in two modes:
- **Detection**: log attacks but don't block
- **Prevention**: block attacks that match OWASP rules

```bash
# Create WAF policy
az network application-gateway waf-policy create \
  -g rg-prod -n waf-policy --type OWASP --version 3.2

# Set to prevention mode
az network application-gateway waf-policy policy-setting update \
  -g rg-prod --policy-name waf-policy --mode Prevention --state Enabled
```

## Azure Front Door

Global L7 load balancer + CDN + WAF at the edge.

| Feature | Front Door | Application Gateway |
|---------|-----------|---------------------|
| Scope | Global (edge POPs) | Regional |
| CDN | ✅ built-in | ❌ |
| WAF | ✅ | ✅ |
| Private Link origin | ✅ (Premium) | ✅ |
| SSL offload | ✅ | ✅ |
| Use case | Global web apps | Single-region web apps |

**Exam tip**: Front Door WAF rules apply at the **edge** (before traffic reaches your origin). App Gateway WAF rules apply at the **region** level.

## DDoS Protection

```bash
# Create DDoS Protection Plan
az network ddos-protection create -g rg-prod -n ddos-plan

# Associate with VNet
az network vnet update -g rg-prod -n vnet-prod \
  --ddos-protection-plan ddos-plan --ddos-protection true
```

**When to recommend Standard over Basic**:
- You have public IPs that need protection
- You need DDoS attack analytics and metrics
- You want the DDoS Rapid Response (DRR) team
- You need **cost protection** (Azure credits back costs incurred by a DDoS attack)

---
## Summary

| Service | Key exam fact |
|---------|---------------|
| **Azure Firewall** | L3-L7, FQDN filtering, threat intel. Premium adds TLS inspection + IDPS. |
| **Firewall Manager** | Hierarchical policies, secured virtual hubs. |
| **App Gateway** | L7 load balancer + WAF. Regional. |
| **Front Door** | Global L7 + CDN + WAF at edge. Premium for Private Link origins. |
| **WAF** | Detection vs Prevention mode. OWASP rule sets. |
| **DDoS Standard** | Per-VNet, analytics, DRR team, cost protection. |

**Next lab**: [03 — Secure Compute, Storage, and Databases](../../03-compute-storage-databases/)